In [ ]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the JSON file
file_path = 'news_json_20260125.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

# Drop 'topic' and 'topic_ko' columns
df = df.drop(columns=['topic', 'topic_ko'], errors='ignore')

# Sort by 'impactScore' in descending order
df = df.sort_values(by='impactScore', ascending=False)

print("Data loaded and cleaned. Shape:", df.shape)
df.head()

In [ ]:
# [NEW] Save the cleaned DataFrame to a new JSON file (news_json_202601.json)
# This saves the state BEFORE adding 'combined_text' or other temporary columns
output_file = 'news_json_202601.json'
df.to_json(output_file, orient='records', force_ascii=False, indent=4)
print(f"Saved cleaned data to {output_file}")

In [ ]:
# 2. Duplicate Detection (Optional)
# Prepare Text for Embedding
df['combined_text'] = df['title'].fillna('') + " " + \
                      df['title_ko'].fillna('') + " " + \
                      df['summary'].fillna('') + " " + \
                      df['summary_ko'].fillna('')

try:
    # Try Deep Learning Embedding (Sentence-Transformer)
    print("Trying SentenceTransformer...")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)
    print("✅ Model loaded successfully!")
except ImportError:
    # Fallback to TF-IDF
    print("⚠️ SentenceTransformer not found. Switching to TF-IDF (scikit-learn)...")
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer(max_features=5000)
    embeddings = vectorizer.fit_transform(df['combined_text'].tolist())
    print("✅ TF-IDF calculated successfully!")

# Calculate Cosine Similarity
similarity_matrix = cosine_similarity(embeddings)

# Find Duplicates
threshold = 0.85 
duplicates = []
num_articles = len(df)

for i in range(num_articles):
    for j in range(i + 1, num_articles):
        score = similarity_matrix[i][j]
        if score >= threshold:
            item_i = df.iloc[i]
            item_j = df.iloc[j]
            duplicates.append({
                "score": score,
                "id_1": item_i['id'],
                "title_1": item_i['title'],
                "id_2": item_j['id'],
                "title_2": item_j['title']
            })

print(f"\nFound {len(duplicates)} pairs of potential duplicates (Threshold: {threshold}):\n")

for d in sorted(duplicates, key=lambda x: x['score'], reverse=True):
    print(f"Similarity: {d['score']:.4f}")
    print(f"1: {d['title_1']} ({d['id_1']})")
    print(f"2: {d['title_2']} ({d['id_2']})")
    print("-" * 50)